# Medical RAG MVP

Local retrieval-augmented generation for healthcare information. This notebook
is **Path A** in the Baymax repo: the original prototype that grew into the
production stack (`hiro` API + `bashmax` terminal client).

**Repository:** [https://github.com/ArshiAbolghasemi/Baymax](https://github.com/ArshiAbolghasemi/Baymax)

Follow **Path A** in the root `README.md` if you cloned from GitHub: install
`ipykernel`, open this file, select a Python kernel, run from the top, and
leave `SKIP_LLM = True` unless you have a GPU and a Hugging Face token.

**This is not a doctor.** Answers can be incomplete or wrong. Do not use them
for diagnosis, dosing, or emergency care. For anything urgent, contact a
clinician or emergency services.

## Data sources

| Source | What it contributes |
| --- | --- |
| [DailyMed](https://dailymed.nlm.nih.gov/) | Official US prescribing labels (SPL) |
| [MedlinePlus](https://medlineplus.gov/) | Patient-friendly health topics (NLM search API) |
| [openFDA](https://open.fda.gov/) | FDA drug-label JSON |
| [Medicib](https://medicib.com/blog) | Traditional-medicine articles (optional crawl) |

ITMA, JIITM, and CMJA were listed in early drafts. They are left in
`CONFIG["optional_crawl_sites"]` but are off by default so a first run does
not hammer those hosts.

## Pipeline

```
Public APIs / crawl  →  clean  →  chunk  →  E5 embeddings  →  ChromaDB
                                                              ↓
Question  →  retriever  →  MedGemma (chat template)  →  cited answer
```

The production service in this repo does **not** use this notebook at runtime.
It serves the same sources through LangGraph tools (DailyMed, MedlinePlus,
openFDA/FAERS, Genetics) plus an optional Qdrant knowledge base. See the
root `README.md` to run that stack.

## What this rewrite fixes

The previous notebook could not run end-to-end. The main defects were:

- Class methods (`search`, `get_label`, `to_document`, …) were defined as
  free functions, so `loader.search(...)` raised `AttributeError`
- DailyMed loading stopped at a literal `...` and never built documents
- `openfda_docs` and `medline_docs` were never created
- `HuggingFaceEmbeddings` and `Chroma` were used without imports
- Crawled pages had no `source` metadata, so later cells hit `KeyError`
- `extract_text` can return `None`; `crawl_article` passed that into `Document`
- The crawler downloaded every page twice
- `normalize_text` collapsed newlines with `\s+` before it tried to tidy them
- `db.persist()` is gone in current langchain-chroma (disk writes are automatic)
- MedGemma was called as raw `text-generation`, so the prompt was echoed back
- `do_sample=False` was combined with a temperature, which Transformers rejects
- CUDA device name was read even when no GPU existed
- Gradio `ChatInterface` was given a one-argument function
- `medical_chat("What is Aspirin?")` sat *inside* the function after `return`
- Leftover cells referenced undefined names (`item`, incomplete loops)

## 1. Install

**Before this cell:** in a terminal, `python -m pip install ipykernel notebook`,
then in the editor pick **Select Kernel → Python 3**.

Run this cell once per kernel. `%pip` installs into the notebook's own
environment (Jupyter, VS Code, Cursor, or Colab). Restart the kernel if
imports still fail.

In [1]:
%pip install -q \
    "langchain>=0.3" \
    "langchain-core>=0.3" \
    "langchain-text-splitters>=0.3" \
    "langchain-chroma>=0.2" \
    chromadb \
    sentence-transformers \
    beautifulsoup4 \
    lxml \
    trafilatura \
    requests \
    pandas \
    numpy \
    tqdm \
    gradio \
    python-dotenv \
    transformers \
    accelerate \
    sentencepiece \
    huggingface_hub

Note: you may need to restart the kernel to use updated packages.


## 2. Imports, configuration, logging

In [2]:
from __future__ import annotations

import hashlib
import json
import logging
import os
import re
import time
import xml.etree.ElementTree as ET
from collections import Counter, deque
from pathlib import Path
from urllib.parse import urldefrag, urljoin, urlparse
from urllib.robotparser import RobotFileParser

import numpy as np
import pandas as pd
import requests
import torch
import trafilatura
from bs4 import BeautifulSoup
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("MedicalRAG")

ROOT = Path.cwd()
DATA_DIR = ROOT / "data" / "notebook"
DATA_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "embedding_model": "intfloat/multilingual-e5-base",
    "chunk_size": 1000,
    "chunk_overlap": 200,
    "top_k": 5,
    "crawl_depth": 2,
    "request_timeout": 20,
    "delay_between_requests": 1.5,
    "max_pages_per_site": 12,
    "min_page_chars": 200,
    "min_doc_chars": 300,
    "vector_db": str(DATA_DIR / "medical_db"),
    "collection_name": "medical_rag",
    "medgemma_model": "google/medgemma-4b-it",
    "max_new_tokens": 512,
    # Traditional-medicine crawl. Keep this short for a first run.
    "crawl_sites": ["https://medicib.com/blog"],
    "optional_crawl_sites": [
        "https://www.itma.ir/",
        "https://jiitm.ir/",
        "https://cmja.arakmu.ac.ir/",
    ],
    "dailymed_drugs": [
        "aspirin",
        "ibuprofen",
        "acetaminophen",
        "metformin",
        "losartan",
    ],
    "openfda_drugs": [
        "aspirin",
        "ibuprofen",
        "acetaminophen",
        "metformin",
        "losartan",
    ],
    "medlineplus_queries": [
        "hypertension",
        "diabetes",
        "aspirin",
        "fever",
        "headache",
    ],
}

DAILYMED_API = "https://dailymed.nlm.nih.gov/dailymed/services/v2"
OPENFDA_LABEL_URL = "https://api.fda.gov/drug/label.json"
MEDLINEPLUS_SEARCH_URL = "https://wsearch.nlm.nih.gov/ws/query"

session = requests.Session()
session.headers.update(
    {
        "User-Agent": "Baymax-Medical-RAG/1.0 (research; contact via repository)",
        "Accept": "text/html,application/json,application/xml;q=0.9,*/*;q=0.8",
    }
)

logger.info("Notebook started. Data directory: %s", DATA_DIR)

2026-08-16 00:38:51,505 | INFO | Notebook started. Data directory: C:\Users\DELL\Desktop\Drug\Baymax\data\notebook


## 3. HTTP helpers and polite crawler

`can_fetch` caches `robots.txt` per host. `crawl_site` reuses the HTML it
already downloaded instead of fetching every URL twice, caps visits (not only
kept documents), and skips non-http(s) links, fragments, and binary files.
Every document carries a `source` field so later cells never KeyError.

In [3]:
SKIP_EXTENSIONS = {
    ".pdf", ".jpg", ".jpeg", ".png", ".gif", ".svg", ".webp",
    ".zip", ".mp4", ".mp3", ".css", ".js", ".ico", ".woff", ".woff2",
}
_robots_cache: dict[str, RobotFileParser | None] = {}


def can_fetch(url: str) -> bool:
    parsed = urlparse(url)
    origin = f"{parsed.scheme}://{parsed.netloc}"
    if origin not in _robots_cache:
        rp = RobotFileParser()
        rp.set_url(f"{origin}/robots.txt")
        try:
            rp.read()
            _robots_cache[origin] = rp
        except Exception as exc:
            logger.debug("robots.txt unavailable for %s: %s", origin, exc)
            _robots_cache[origin] = None
    rp = _robots_cache[origin]
    if rp is None:
        return True
    return rp.can_fetch(session.headers["User-Agent"], url)


def download_html(url: str) -> str | None:
    try:
        response = session.get(url, timeout=CONFIG["request_timeout"])
        if response.status_code != 200:
            logger.warning("HTTP %s for %s", response.status_code, url)
            return None
        return response.text
    except requests.RequestException as exc:
        logger.warning("download failed %s: %s", url, exc)
        return None


def extract_text(html: str | None) -> str | None:
    if not html:
        return None
    extracted = trafilatura.extract(
        html,
        include_links=False,
        include_images=False,
        include_tables=True,
    )
    return extracted or None


def normalize_url(url: str) -> str:
    url, _frag = urldefrag(url)
    return url.rstrip("/") or url


def is_crawlable(url: str, base_netloc: str) -> bool:
    parsed = urlparse(url)
    if parsed.scheme not in {"http", "https"}:
        return False
    if parsed.netloc != base_netloc:
        return False
    path = parsed.path.lower()
    return not any(path.endswith(ext) for ext in SKIP_EXTENSIONS)


def get_links(html: str, base_url: str) -> list[str]:
    soup = BeautifulSoup(html, "lxml")
    base_netloc = urlparse(base_url).netloc
    links: set[str] = set()
    for tag in soup.find_all("a", href=True):
        href = normalize_url(urljoin(base_url, tag["href"]))
        if is_crawlable(href, base_netloc):
            links.add(href)
    return list(links)


def content_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


visited_hashes: set[str] = set()


def is_duplicate(text: str) -> bool:
    digest = content_hash(text)
    if digest in visited_hashes:
        return True
    visited_hashes.add(digest)
    return False


def html_to_document(url: str, html: str, source: str) -> Document | None:
    text = extract_text(html)
    if text is None or len(text) < CONFIG["min_page_chars"]:
        return None
    if is_duplicate(text):
        return None
    soup = BeautifulSoup(html, "lxml")
    title = soup.title.get_text(strip=True) if soup.title else ""
    return Document(
        page_content=text,
        metadata={"url": url, "title": title, "source": source},
    )


def crawl_site(start_url: str, source: str = "Traditional Medicine") -> list[Document]:
    visited: set[str] = set()
    queue: deque[tuple[str, int]] = deque([(normalize_url(start_url), 0)])
    documents: list[Document] = []
    pages = 0

    while queue and pages < CONFIG["max_pages_per_site"]:
        url, depth = queue.popleft()
        if url in visited or depth > CONFIG["crawl_depth"]:
            continue
        visited.add(url)
        if not can_fetch(url):
            logger.info("robots.txt disallows %s", url)
            continue

        logger.info("Crawling depth=%s URL=%s", depth, url)
        html = download_html(url)
        pages += 1
        if html is None:
            continue

        document = html_to_document(url, html, source)
        if document:
            documents.append(document)

        if depth < CONFIG["crawl_depth"]:
            for link in get_links(html, url):
                if link not in visited:
                    queue.append((link, depth + 1))

        time.sleep(CONFIG["delay_between_requests"])

    logger.info("Crawl finished %s: %s documents from %s pages", start_url, len(documents), pages)
    return documents

## 4. Traditional-medicine crawl

Set `SKIP_CRAWL = True` to reuse `data/notebook/traditional_medicine.json` if
you already crawled. Optional Iranian journal sites stay off unless you add
them to `CONFIG["crawl_sites"]`.

In [4]:
SKIP_CRAWL = False
CRAWL_CACHE = DATA_DIR / "traditional_medicine.json"

all_documents: list[Document] = []

if SKIP_CRAWL and CRAWL_CACHE.exists():
    raw_crawl = json.loads(CRAWL_CACHE.read_text(encoding="utf-8"))
    all_documents = [
        Document(page_content=row["content"], metadata={k: v for k, v in row.items() if k != "content"})
        for row in raw_crawl
    ]
    logger.info("Loaded %s cached crawl documents", len(all_documents))
else:
    for website in CONFIG["crawl_sites"]:
        logger.info("Starting crawl %s", website)
        all_documents.extend(crawl_site(website, source="Traditional Medicine"))

print(f"Traditional-medicine documents: {len(all_documents)}")
if all_documents:
    print(all_documents[0].metadata)
    print(all_documents[0].page_content[:500])

2026-08-16 00:38:51,564 | INFO | Starting crawl https://medicib.com/blog
2026-08-16 00:38:52,808 | INFO | robots.txt disallows https://medicib.com/blog
2026-08-16 00:38:52,810 | INFO | Crawl finished https://medicib.com/blog: 0 documents from 0 pages


Traditional-medicine documents: 0


Traditional-medicine documents: 0


## 5. DailyMed — official US labels

Search returns SPL setids. The label itself is XML; we pull the sections that
matter for a patient-facing assistant instead of dumping the raw JSON tree.

In [5]:
LABEL_SECTIONS = {
    "indications": ("indications and usage", "indications"),
    "dosage": ("dosage and administration", "dosage"),
    "contraindications": ("contraindications",),
    "warnings": ("warnings and precautions", "boxed warning", "warnings"),
    "adverse_reactions": ("adverse reactions",),
    "drug_interactions": ("drug interactions",),
}


def _local(tag: str) -> str:
    return tag.rsplit("}", 1)[-1]


def _element_text(element: ET.Element | None, limit: int = 4000) -> str:
    if element is None:
        return ""
    text = " ".join(element.itertext())
    text = re.sub(r"\s+", " ", text).strip()
    return text[:limit]


class DailyMedLoader:
    def __init__(self, base: str = DAILYMED_API) -> None:
        self.base = base.rstrip("/")

    def search(self, keyword: str, pagesize: int = 5) -> list[dict]:
        url = f"{self.base}/spls.json"
        try:
            response = session.get(
                url,
                params={"drug_name": keyword, "name_type": "both", "pagesize": pagesize, "page": 1},
                timeout=CONFIG["request_timeout"],
            )
            response.raise_for_status()
            return response.json().get("data") or []
        except (requests.RequestException, ValueError) as exc:
            logger.warning("DailyMed search failed for %s: %s", keyword, exc)
            return []

    def get_label_xml(self, setid: str) -> str | None:
        url = f"{self.base}/spls/{setid}.xml"
        try:
            response = session.get(url, timeout=CONFIG["request_timeout"])
            response.raise_for_status()
            return response.text
        except requests.RequestException as exc:
            logger.warning("DailyMed label failed for %s: %s", setid, exc)
            return None

    def spl_to_text(self, xml_text: str, title: str) -> str:
        try:
            root = ET.fromstring(xml_text)
        except ET.ParseError as exc:
            logger.warning("DailyMed XML parse error: %s", exc)
            return title

        found: dict[str, list[str]] = {name: [] for name in LABEL_SECTIONS}
        for section in root.iter():
            if _local(section.tag) != "section":
                continue
            heading = ""
            for child in section:
                if _local(child.tag) == "title":
                    heading = _element_text(child, 200).casefold()
                    break
            if not heading:
                continue
            for name, terms in LABEL_SECTIONS.items():
                if any(term in heading for term in terms):
                    body = _element_text(section)
                    if body and body not in found[name]:
                        found[name].append(body)

        parts = [title]
        for name, chunks in found.items():
            if chunks:
                parts.append(f"{name.replace('_', ' ').title()}: " + " ".join(chunks))
        return "\n\n".join(parts)

    def to_document(self, keyword: str, entry: dict, xml_text: str) -> Document:
        setid = str(entry.get("setid", ""))
        title = str(entry.get("title") or keyword)
        url = f"https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid={setid}"
        return Document(
            page_content=self.spl_to_text(xml_text, title),
            metadata={
                "source": "DailyMed",
                "url": url,
                "title": title,
                "setid": setid,
                "drug": keyword,
            },
        )

    def load_drugs(self, drugs: list[str], per_drug: int = 1) -> list[Document]:
        documents: list[Document] = []
        for drug in tqdm(drugs, desc="DailyMed"):
            hits = self.search(drug)
            for entry in hits[:per_drug]:
                xml_text = self.get_label_xml(str(entry.get("setid", "")))
                if not xml_text:
                    continue
                documents.append(self.to_document(drug, entry, xml_text))
                time.sleep(CONFIG["delay_between_requests"])
        return documents


dailymed_loader = DailyMedLoader()
dailymed_docs = dailymed_loader.load_drugs(CONFIG["dailymed_drugs"])
print(f"DailyMed documents: {len(dailymed_docs)}")
if dailymed_docs:
    print(dailymed_docs[0].metadata)
    print(dailymed_docs[0].page_content[:600])

DailyMed:   0%|          | 0/5 [00:00<?, ?it/s]

DailyMed documents: 5
{'source': 'DailyMed', 'url': 'https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=54e69d5a-55d7-4ebe-8494-0be93613f1e2', 'title': 'ASPIRIN 81MG (ASPIRIN) TABLET, DELAYED RELEASE [HARRIS TEETER]', 'setid': '54e69d5a-55d7-4ebe-8494-0be93613f1e2', 'drug': 'aspirin'}
ASPIRIN 81MG (ASPIRIN) TABLET, DELAYED RELEASE [HARRIS TEETER]

Warnings: Warnings Reye's syndrome Children and teenagers who have or are recovering from chicken pox or flu-like symptoms should not use this product. When using this product, if changes in behavior with nausea and vomiting occur, consult a doctor because these symptoms could be an early sign of Reye’s syndrome, a rare but serious illness. Allergy alert Aspirin may cause a severe allergic reaction which may include: hives facial swelling asthma (wheezing) shock Stomach bleeding warning This product contains an NSAID, which may ca


## 6. openFDA — FDA drug-label JSON

Query by generic name. Each result is flattened into readable sections rather
than `json.dumps` of the whole object (which produced huge, noisy chunks).

In [6]:
OPENFDA_FIELDS = [
    ("indications_and_usage", "Indications"),
    ("dosage_and_administration", "Dosage"),
    ("contraindications", "Contraindications"),
    ("warnings", "Warnings"),
    ("adverse_reactions", "Adverse reactions"),
    ("drug_interactions", "Drug interactions"),
]


class OpenFDALoader:
    def __init__(self, url: str = OPENFDA_LABEL_URL) -> None:
        self.url = url

    def search(self, keyword: str, limit: int = 3) -> list[dict]:
        query = f'openfda.generic_name:"{keyword}"'
        try:
            response = session.get(
                self.url,
                params={"search": query, "limit": limit},
                timeout=CONFIG["request_timeout"],
            )
            if response.status_code == 404:
                return []
            response.raise_for_status()
            return response.json().get("results") or []
        except (requests.RequestException, ValueError) as exc:
            logger.warning("openFDA search failed for %s: %s", keyword, exc)
            return []

    def to_document(self, keyword: str, item: dict) -> Document | None:
        openfda = item.get("openfda") or {}
        brand = ", ".join(openfda.get("brand_name") or []) or keyword
        generic = ", ".join(openfda.get("generic_name") or []) or keyword
        parts = [f"{brand} ({generic})"]
        for field, heading in OPENFDA_FIELDS:
            values = item.get(field) or []
            if values:
                parts.append(f"{heading}: " + " ".join(str(v) for v in values[:2]))
        text = "\n\n".join(parts)
        if len(text) < CONFIG["min_page_chars"]:
            return None
        setid = (openfda.get("spl_set_id") or [""])[0]
        return Document(
            page_content=text,
            metadata={
                "source": "openFDA",
                "url": f"https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid={setid}" if setid else "https://open.fda.gov/",
                "title": brand,
                "drug": keyword,
            },
        )

    def load_drugs(self, drugs: list[str]) -> list[Document]:
        documents: list[Document] = []
        for drug in tqdm(drugs, desc="openFDA"):
            for item in self.search(drug):
                doc = self.to_document(drug, item)
                if doc:
                    documents.append(doc)
            time.sleep(0.4)
        return documents


openfda_loader = OpenFDALoader()
openfda_docs = openfda_loader.load_drugs(CONFIG["openfda_drugs"])
print(f"openFDA documents: {len(openfda_docs)}")

openFDA:   0%|          | 0/5 [00:00<?, ?it/s]

openFDA documents: 15


## 7. MedlinePlus — NLM search API

The website URL is not an API. The production Baymax tools use
`https://wsearch.nlm.nih.gov/ws/query` (`db=healthTopics`). This notebook
does the same and parses the XML `document` list.

In [7]:
class MedlineLoader:
    def __init__(self, search_url: str = MEDLINEPLUS_SEARCH_URL) -> None:
        self.search_url = search_url

    def search(self, query: str, retmax: int = 5) -> list[dict]:
        try:
            response = session.get(
                self.search_url,
                params={
                    "db": "healthTopics",
                    "term": query,
                    "retmax": retmax,
                    "rettype": "brief",
                    "tool": "baymax-notebook",
                },
                timeout=CONFIG["request_timeout"],
            )
            response.raise_for_status()
        except requests.RequestException as exc:
            logger.warning("MedlinePlus search failed for %s: %s", query, exc)
            return []

        try:
            root = ET.fromstring(response.text)
        except ET.ParseError as exc:
            logger.warning("MedlinePlus XML parse error: %s", exc)
            return []

        results = []
        for document in root.findall(".//document")[:retmax]:
            fields: dict[str, list[str]] = {}
            for content in document.findall("./content"):
                name = content.attrib.get("name", "").lower()
                fields.setdefault(name, []).append("".join(content.itertext()).strip())
            title = next(iter(fields.get("title", [])), "").strip()
            summary = next(
                iter(fields.get("fullsummary", []) or fields.get("snippet", [])),
                "",
            ).strip()
            url = document.attrib.get("url", "")
            if title or url:
                results.append({"title": title, "summary": summary, "url": url})
        return results

    def to_document(self, query: str, hit: dict) -> Document | None:
        summary = hit.get("summary") or ""
        title = hit.get("title") or query
        if len(summary) < 40:
            return None
        return Document(
            page_content=f"{title}\n\n{summary}",
            metadata={
                "source": "MedlinePlus",
                "url": hit.get("url") or "https://medlineplus.gov/",
                "title": title,
                "query": query,
            },
        )

    def load_queries(self, queries: list[str]) -> list[Document]:
        documents: list[Document] = []
        for query in tqdm(queries, desc="MedlinePlus"):
            for hit in self.search(query):
                doc = self.to_document(query, hit)
                if doc:
                    documents.append(doc)
            time.sleep(0.4)
        return documents


medline_loader = MedlineLoader()
medline_docs = medline_loader.load_queries(CONFIG["medlineplus_queries"])
print(f"MedlinePlus documents: {len(medline_docs)}")

MedlinePlus:   0%|          | 0/5 [00:00<?, ?it/s]

MedlinePlus documents: 25


## 8. Merge, clean, and persist the corpus

In [8]:
documents: list[Document] = []
documents.extend(all_documents)
documents.extend(dailymed_docs)
documents.extend(openfda_docs)
documents.extend(medline_docs)

print("Raw documents:", len(documents))
print(Counter(doc.metadata.get("source", "unknown") for doc in documents))


def normalize_text(text: str | None) -> str:
    if not text:
        return ""
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[^\S\n]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    return text.strip()


clean_documents: list[Document] = []
for doc in documents:
    content = normalize_text(doc.page_content)
    if len(content) < CONFIG["min_doc_chars"]:
        continue
    metadata = dict(doc.metadata)
    metadata.setdefault("source", "unknown")
    metadata.setdefault("url", "")
    metadata.setdefault("title", "")
    # Chroma metadata values must be scalars.
    metadata = {k: (v if isinstance(v, (str, int, float, bool)) else str(v)) for k, v in metadata.items()}
    clean_documents.append(Document(page_content=content, metadata=metadata))

seen: set[str] = set()
unique: list[Document] = []
for doc in clean_documents:
    digest = content_hash(doc.page_content)
    if digest in seen:
        continue
    seen.add(digest)
    unique.append(doc)

print("Unique documents:", len(unique))
print(Counter(doc.metadata["source"] for doc in unique))
if unique:
    lengths = [len(doc.page_content) for doc in unique]
    print(f"chars mean={np.mean(lengths):.0f} min={min(lengths)} max={max(lengths)}")

raw = [
    {
        "url": doc.metadata.get("url", ""),
        "title": doc.metadata.get("title", ""),
        "source": doc.metadata.get("source", ""),
        "content": doc.page_content,
    }
    for doc in unique
]
pd.DataFrame(raw).to_csv(DATA_DIR / "corpus.csv", index=False)
(DATA_DIR / "corpus.json").write_text(
    json.dumps(raw, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
if all_documents:
    crawl_raw = [
        {
            "url": doc.metadata.get("url", ""),
            "title": doc.metadata.get("title", ""),
            "source": doc.metadata.get("source", "Traditional Medicine"),
            "content": doc.page_content,
        }
        for doc in all_documents
    ]
    CRAWL_CACHE.write_text(json.dumps(crawl_raw, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote", DATA_DIR / "corpus.csv")

Raw documents: 45
Counter({'MedlinePlus': 25, 'openFDA': 15, 'DailyMed': 5})
Unique documents: 45
Counter({'MedlinePlus': 25, 'openFDA': 15, 'DailyMed': 5})
chars mean=6010 min=784 max=20642
Wrote C:\Users\DELL\Desktop\Drug\Baymax\data\notebook\corpus.csv


## 9. Chunking

`RecursiveCharacterTextSplitter` keeps paragraphs together before falling
back to sentences and words. Chunk size comes from `CONFIG`, not a hardcoded
copy of those numbers.

In [9]:
if not unique:
    raise RuntimeError(
        "No documents survived cleaning. Check network access to DailyMed / "
        "MedlinePlus / openFDA, or disable SKIP_CRAWL and crawl a site."
    )

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["chunk_size"],
    chunk_overlap=CONFIG["chunk_overlap"],
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(unique)
print(f"Chunks: {len(chunks)}")
print(chunks[0].metadata)
print(chunks[0].page_content[:400])
chunk_lengths = [len(c.page_content) for c in chunks]
print(f"chunk chars mean={np.mean(chunk_lengths):.0f} min={min(chunk_lengths)} max={max(chunk_lengths)}")

Chunks: 412
{'source': 'DailyMed', 'url': 'https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=54e69d5a-55d7-4ebe-8494-0be93613f1e2', 'title': 'ASPIRIN 81MG (ASPIRIN) TABLET, DELAYED RELEASE [HARRIS TEETER]', 'setid': '54e69d5a-55d7-4ebe-8494-0be93613f1e2', 'drug': 'aspirin'}
ASPIRIN 81MG (ASPIRIN) TABLET, DELAYED RELEASE [HARRIS TEETER]
chunk chars mean=712 min=5 max=1000


## 10. Embeddings and ChromaDB

`intfloat/multilingual-e5-base` expects `query: ` / `passage: ` prefixes.
Without them retrieval quality collapses. Embeddings are L2-normalised so
Chroma cosine/dot-product search is well behaved.

langchain-chroma writes to `persist_directory` as you upsert. There is no
`db.persist()` in current versions.

In [10]:
class E5Embeddings(Embeddings):
    """Sentence-Transformers wrapper with the E5 query/passage prefixes."""

    def __init__(self, model_name: str, device: str | None = None) -> None:
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(model_name, device=device)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        prefixed = [f"passage: {text}" for text in texts]
        vectors = self.model.encode(prefixed, normalize_embeddings=True, show_progress_bar=True)
        return vectors.tolist()

    def embed_query(self, text: str) -> list[float]:
        vector = self.model.encode(f"query: {text}", normalize_embeddings=True)
        return vector.tolist()


embedding = E5Embeddings(CONFIG["embedding_model"])
probe = embedding.embed_query("What is aspirin?")
print("Embedding dimensions:", len(probe))

db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory=CONFIG["vector_db"],
    collection_name=CONFIG["collection_name"],
)
print("Vector store ready at", CONFIG["vector_db"])

retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": CONFIG["top_k"], "fetch_k": 20},
)

print("\nSimilarity search: What is hypertension?")
for doc in db.similarity_search("What is hypertension?", k=3):
    print("=" * 60)
    print(doc.metadata)
    print(doc.page_content[:400])

print("\nMMR retriever: What is metformin?")
for doc in retriever.invoke("What is metformin?"):
    print("=" * 60)
    print(doc.metadata)
    print(doc.page_content[:400])

2026-08-16 00:39:24,731 | INFO | Load pretrained SentenceTransformer: intfloat/multilingual-e5-base


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding dimensions: 768


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Vector store ready at C:\Users\DELL\Desktop\Drug\Baymax\data\notebook\medical_db

Similarity search: What is hypertension?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{'query': 'hypertension', 'url': 'https://medlineplus.gov/howtopreventhighbloodpressure.html', 'source': 'MedlinePlus', 'title': 'How to Prevent <span class="qt0"><span class="qt1">High Blood Pressure</span></span>'}
<p>Around half of American adults have <span class="qt0"><span class="qt1">high blood pressure (hypertension</span></span>). Many of those people don't know they have it because there are usually no warning signs. This can be dangerous, because <span class="qt0"><span class="qt1">high blood pressure</span></span> can lead to life-threatening conditions like heart attack or stroke. The good news is
{'query': 'hypertension', 'url': 'https://medlineplus.gov/howtopreventhighbloodpressure.html', 'title': 'How to Prevent <span class="qt0"><span class="qt1">High Blood Pressure</span></span>', 'source': 'MedlinePlus'}
<p>Around half of American adults have <span class="qt0"><span class="qt1">high blood pressure (hypertension</span></span>). Many of those people don't know they hav

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{'source': 'openFDA', 'title': 'Metformin hydrochloride', 'url': 'https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=014f2a32-8440-4d0c-8f5d-c4b3a49e0dec', 'drug': 'metformin'}
Metformin hydrochloride (METFORMIN HYDROCHLORIDE)

Indications: 1 INDICATIONS AND USAGE Metformin hydrochloride extended-release tablets, USP are indicated as an adjunct to diet and exercise to improve glycemic control in adults with type 2 diabetes mellitus. Metformin hydrochloride extended-release tablets, USP are a biguanide indicated as an adjunct to diet and exercise to improve glycemic contr
{'title': 'Metformin hydrochloride', 'drug': 'metformin', 'source': 'openFDA', 'url': 'https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=014f2a32-8440-4d0c-8f5d-c4b3a49e0dec'}
. Cholestatic, hepatocellular, and mixed hepatocellular liver injury have been reported with postmarketing use of metformin.
{'drug': 'metformin', 'source': 'openFDA', 'title': 'Metformin Hydrochloride', 'url': 'https://dailymed.nlm.

## 11. MedGemma

`google/medgemma-4b-it` is a **gated** Hugging Face model. Accept the terms on
the [model card](https://huggingface.co/google/medgemma-4b-it) and set
`HF_TOKEN` (or run `huggingface-cli login`) before loading.

The original notebook used a `text-generation` pipeline on a concatenated
prompt, which (a) echoed the prompt back as the "answer" and (b) ignored
Gemma's chat template. We apply the chat template and decode only new tokens.

A 4B model wants a GPU. **`SKIP_LLM = True` is the default** so a laptop
without a GPU can still retrieve passages. On CPU, loading MedGemma uses
float32 and can freeze the machine.

To generate answers: accept the model terms, set `HF_TOKEN`, then change
`SKIP_LLM` to `False`.

In [11]:
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer

SKIP_LLM = True  # set False only if you have GPU + HF_TOKEN
MODEL_NAME = CONFIG["medgemma_model"]

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)

has_cuda = torch.cuda.is_available()
print("CUDA available:", has_cuda)
if has_cuda:
    print("GPU:", torch.cuda.get_device_name(0))

tokenizer = None
model = None

if SKIP_LLM:
    logger.warning("SKIP_LLM=True — retrieval will work, generation will return a stub.")
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    dtype = torch.bfloat16 if has_cuda and torch.cuda.is_bf16_supported() else (
        torch.float16 if has_cuda else torch.float32
    )
    load_kwargs = {
        "low_cpu_mem_usage": True,
        "device_map": "auto" if has_cuda else None,
    }
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype, **load_kwargs)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, **load_kwargs)
    if not has_cuda:
        model = model.to("cpu")
    model.eval()
    print("Loaded", MODEL_NAME, "dtype=", dtype)

2026-08-16 00:44:11,383 | WARNING | SKIP_LLM=True — retrieval will work, generation will return a stub.


CUDA available: False


## 12. RAG: retrieve → prompt → generate

In [12]:
SYSTEM_PROMPT = """You are a careful medical assistant.

Use ONLY the supplied context. Never invent drug names, doses, or sources.
If the context does not contain the answer, say you don't know.

Always mention the source of each fact (DailyMed, MedlinePlus, openFDA, or
Traditional Medicine).

Priority when sources disagree:
1. DailyMed
2. MedlinePlus
3. openFDA
4. Traditional Medicine — label it clearly as traditional medicine, not as
   established clinical evidence.

This is educational information, not a diagnosis or a prescription.
Recommend seeing a clinician for anything urgent."""


def retrieve(question: str) -> list[Document]:
    return retriever.invoke(question)


def build_messages(question: str, docs: list[Document]) -> list[dict[str, str]]:
    blocks = []
    for index, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "unknown")
        url = doc.metadata.get("url", "")
        blocks.append(f"[{index}] Source: {source}\nURL: {url}\n{doc.page_content}")
    context = "\n\n".join(blocks) if blocks else "(no retrieved context)"
    user = (
        f"Context:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        "Answer in the same language as the question. Cite sources by name."
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]


def ask_llm(question: str, docs: list[Document]) -> str:
    if model is None or tokenizer is None:
        sources = ", ".join(sorted({d.metadata.get("source", "?") for d in docs})) or "none"
        return (
            f"[LLM skipped] Retrieved {len(docs)} passage(s) from {sources}. "
            "Set SKIP_LLM=False and load MedGemma to generate an answer."
        )

    messages = build_messages(question, docs)
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def medical_chat(question: str) -> dict:
    docs = retrieve(question)
    answer = ask_llm(question, docs)
    return {"answer": answer, "documents": docs}


def pretty_print(result: dict) -> None:
    print("=" * 80)
    print(result["answer"])
    print()
    print("=" * 80)
    print("Sources")
    for doc in result["documents"]:
        print(f"- {doc.metadata.get('source')} | {doc.metadata.get('title', '')} | {doc.metadata.get('url', '')}")


demo_result = medical_chat("What is aspirin used for?")
pretty_print(demo_result)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[LLM skipped] Retrieved 5 passage(s) from DailyMed, openFDA. Set SKIP_LLM=False and load MedGemma to generate an answer.

Sources
- openFDA | Low Dose Aspirin | https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=0058175f-3474-40c3-a046-6cfaec86d84b
- openFDA | Rapidol Aspirin | https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=00d9ab0d-ff25-784f-e063-6294a90a8497
- openFDA | Ibuprofen | https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=00872852-680a-41b3-901b-b991b12a176d
- DailyMed | IBUPROFEN PM (DIPHENHYDRAMINE CITRATE, IBUPROFEN) TABLET, COATED [WALMART INC. (SEE ALSO EQUATE)] | https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=d86151b6-d5c5-4d93-9cfd-43ab7ad0443a
- DailyMed | IBUPROFEN PM (DIPHENHYDRAMINE CITRATE, IBUPROFEN) TABLET, COATED [WALMART INC. (SEE ALSO EQUATE)] | https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=d86151b6-d5c5-4d93-9cfd-43ab7ad0443a


## 13. Example questions

Includes an English clinical question and a Persian traditional-medicine
question so multilingual-e5 has something to retrieve against.

In [13]:
examples = [
    "What is hypertension?",
    "What are side effects of aspirin?",
    "What is diabetes?",
    "What is metformin used for?",
    "در طب سنتی برای سردرد چه توصیه‌ای وجود دارد؟",
]

for question in examples:
    print("\n" + "#" * 80)
    print(question)
    pretty_print(medical_chat(question))


################################################################################
What is hypertension?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[LLM skipped] Retrieved 5 passage(s) from MedlinePlus, openFDA. Set SKIP_LLM=False and load MedGemma to generate an answer.

Sources
- MedlinePlus | How to Prevent <span class="qt0"><span class="qt1">High Blood Pressure</span></span> | https://medlineplus.gov/howtopreventhighbloodpressure.html
- MedlinePlus | Blood Pressure Medicines | https://medlineplus.gov/bloodpressuremedicines.html
- MedlinePlus | Pulmonary <span class="qt0"><span class="qt1">Hypertension</span></span> | https://medlineplus.gov/pulmonaryhypertension.html
- MedlinePlus | <span class="qt0"><span class="qt1">High Blood Pressure</span></span> | https://medlineplus.gov/highbloodpressure.html
- openFDA | Losartan Potassium and Hydrochlorothiazide | https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=03c0ec8d-9daf-43e8-8a4e-a2ddc3573ca7

################################################################################
What are side effects of aspirin?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[LLM skipped] Retrieved 5 passage(s) from MedlinePlus, openFDA. Set SKIP_LLM=False and load MedGemma to generate an answer.

Sources
- MedlinePlus | Drug Reactions | https://medlineplus.gov/drugreactions.html
- openFDA | Ibuprofen Dye Free | https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=00653b7c-7099-487e-9a01-e89781c21323
- openFDA | Rapidol Aspirin | https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=00d9ab0d-ff25-784f-e063-6294a90a8497
- openFDA | Rapidol Aspirin | https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=00d9ab0d-ff25-784f-e063-6294a90a8497
- openFDA | Ibuprofen | https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=00872852-680a-41b3-901b-b991b12a176d

################################################################################
What is diabetes?


...

[LLM skipped] Retrieved 5 passage(s) from MedlinePlus. Set SKIP_LLM=False and load MedGemma to generate an answer.

Sources
- MedlinePlus | <span class="qt0">Diabetes</span> | https://medlineplus.gov/diabetes.html
- MedlinePlus | <span class="qt0">Diabetes</span> Type 1 | https://medlineplus.gov/diabetestype1.html
- MedlinePlus | <span class="qt0">Diabetes</span> Type 1 | https://medlineplus.gov/diabetestype1.html
- MedlinePlus | <span class="qt0">Diabetes</span> Type 2 | https://medlineplus.gov/diabetestype2.html
- MedlinePlus | <span class="qt0">Diabetes</span> | https://medlineplus.gov/diabetes.html

################################################################################
What is metformin used for?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[LLM skipped] Retrieved 5 passage(s) from DailyMed, openFDA. Set SKIP_LLM=False and load MedGemma to generate an answer.

Sources
- openFDA | Metformin hydrochloride | https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=014f2a32-8440-4d0c-8f5d-c4b3a49e0dec
- openFDA | Metformin hydrochloride | https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=014f2a32-8440-4d0c-8f5d-c4b3a49e0dec
- DailyMed | ZITUVIMET (SITAGLIPTIN AND METFORMIN HYDROCHLORIDE) TABLET, FILM COATED [ZYDUS LIFESCIENCES LIMITED] | https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=0098dec4-f0e5-45d5-8aa4-5d0faf9ab142
- DailyMed | ZITUVIMET (SITAGLIPTIN AND METFORMIN HYDROCHLORIDE) TABLET, FILM COATED [ZYDUS LIFESCIENCES LIMITED] | https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=0098dec4-f0e5-45d5-8aa4-5d0faf9ab142
- openFDA | Metformin Hydrochloride | https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=011de1a5-1ac0-4831-9e8d-26ec79ba2205

#####################################################

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[LLM skipped] Retrieved 5 passage(s) from MedlinePlus. Set SKIP_LLM=False and load MedGemma to generate an answer.

Sources
- MedlinePlus | <span class="qt0">Fever</span> | https://medlineplus.gov/fever.html
- MedlinePlus | <span class="qt0">Fever</span> | https://medlineplus.gov/fever.html
- MedlinePlus | How to Prevent <span class="qt0"><span class="qt1">High Blood Pressure</span></span> | https://medlineplus.gov/howtopreventhighbloodpressure.html
- MedlinePlus | Drug Reactions | https://medlineplus.gov/drugreactions.html
- MedlinePlus | Pulmonary <span class="qt0"><span class="qt1">Hypertension</span></span> | https://medlineplus.gov/pulmonaryhypertension.html


## 14. Gradio chat UI

`ChatInterface` calls `fn(message, history)`. A one-argument `ui(question)`
was the original bug. `share=False` keeps the demo local; set `share=True`
only if you intend to expose it.

In [14]:
import gradio as gr


def ui(message, history=None):
    if isinstance(message, dict):
        message = message.get("content", "")
    question = str(message or "").strip()
    if not question:
        return "Please ask a medical question."
    return medical_chat(question)["answer"]


demo = gr.ChatInterface(
    fn=ui,
    title="Medical RAG assistant",
    description=(
        "Prototype RAG over DailyMed, MedlinePlus, openFDA, and optional "
        "traditional-medicine crawl. Not medical advice."
    ),
    examples=[
        "What is hypertension?",
        "What are the warnings on metformin?",
        "What is aspirin used for?",
    ],
)



2026-08-16 00:44:28,306 | INFO | HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"


In [ ]:
demo.launch(debug=True, share=False)

* Running on local URL:  http://127.0.0.1:7860


2026-08-16 00:44:28,045 | INFO | HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-08-16 00:44:28,389 | INFO | HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"


* To create a public link, set `share=True` in `launch()`.


## Reload the vector store later

Re-run the install / import / `E5Embeddings` cells, then:

```python
db = Chroma(
    persist_directory=CONFIG["vector_db"],
    embedding_function=embedding,
    collection_name=CONFIG["collection_name"],
)
retriever = db.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 20})
```

No `db.persist()` call is required.

For the production Baymax assistant (guardrails, ReAct tools, session history,
Qdrant, vLLM), follow the root `README.md` instead of this notebook.